In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import Image, display
import matplotlib.pyplot as plt

In [ ]:
books = pd.read_csv("dataset/Books.csv")
users = pd.read_csv("dataset/Users.csv")
ratings = pd.read_csv("dataset/Ratings.csv")

In [ ]:
# shape of these three datset
print(books.shape)
print(users.shape)
print(ratings.shape)

In [ ]:
books.head()

In [ ]:
ratings.head()

In [ ]:
users.head()

In [ ]:
# Check for missing values in each dataset
print("Missing values in Books Dataset:")
print(books.isnull().sum(), "\n")

print("Missing values in Users Dataset:")
print(users.isnull().sum(), "\n")

print("Missing values in Ratings Dataset:")
print(ratings.isnull().sum())

## Item-based filtering

In [ ]:
ratings_with_book_titles = ratings.merge(books,on='ISBN')

ratings_with_book_titles.drop(columns=["ISBN","Image-URL-S","Image-URL-M"],axis=1,inplace=True)

complete_df = ratings_with_book_titles.merge(users.drop("Age", axis=1), on="User-ID")
complete_df.head()

In [ ]:
complete_df['Location'] = complete_df['Location'].str.split(',').str[-1].str.strip()

complete_df.head()

In [ ]:
# Select user IDs with more than 200 book ratings
min_ratings_threshold = 200

# Count book ratings per user
num_ratings_per_user = complete_df.groupby('User-ID')['Book-Rating'].count()

# Filter users with more than the minimum threshold
knowledgeable_user_ids = num_ratings_per_user[num_ratings_per_user > min_ratings_threshold].index

# Filter ratings from knowledgeable users
knowledgeable_user_ratings = complete_df[complete_df['User-ID'].isin(knowledgeable_user_ids)]

min_ratings_count_threshold=50
rating_counts= knowledgeable_user_ratings.groupby('Book-Title').count()['Book-Rating']
popular_books = rating_counts[rating_counts >= min_ratings_count_threshold].index

final_ratings =  knowledgeable_user_ratings[knowledgeable_user_ratings['Book-Title'].isin(popular_books)]

pt = final_ratings.pivot_table(index='Book-Title',columns='User-ID'
                          ,values='Book-Rating')
pt

In [ ]:
# fill nan with 0
pt.fillna(0,inplace=True)
pt

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# calculate the cosine similarity
similarity_score = cosine_similarity(pt)

In [ ]:
# define the recommender function
def item_recommend(book_name):
    """
    Given a book title, this function returns a list of the top 5 similar books
    based on the collaborative filtering similarity matrix.
    """

    # Step 1: Find the index of the given book in the pivot table
    index = np.where(pt.index == book_name)[0][0]

    # Step 2: Retrieve similarity scores for the given book and sort them
    # sort in descending order and exclude the first result (which is the book itself)
    similar_items = sorted(list(enumerate(similarity_score[index])),
                           key=lambda x: x[1], reverse=True)[1:6]

    # Step 3: Prepare a list to store recommended book details
    recommendations = []

    # Step 4: Loop through the top similar books and extract details
    for i in similar_items:
        book_info = []
        temp_df = books[books['Book-Title'] == pt.index[i[0]]]

        # Extract book title, author, and cover image, ensuring no duplicates
        book_info.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        book_info.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        book_info.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))


        # Append the book info to the recommendations list
        recommendations.append(book_info)

    for book_info in recommendations:
      print("You may like this book.")
      print(f"Title: {book_info[0]}")
      print(f"Author: {book_info[1]}")
      # Display the image
      display(Image(url=book_info[2]))
      print("\n")

In [ ]:
item_recommend('Brave New World')

## User-based filtering

In [ ]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

# Define the rating scale
reader = Reader(rating_scale=(0, 10))

# Load the data into Surprise's dataset format
data = Dataset.load_from_df(complete_df[['User-ID', 'Book-Title', 'Book-Rating']], reader)

# Split the dataset into training and testing sets
train_set, test_set = train_test_split(data, test_size=0.20, random_state=42)

# Define the SVD algorithm
model = SVD()

# Train the algorithm on the training set
model.fit(train_set)

# Make predictions on the test set
predictions = model.test(test_set)

# Evaluate the model
accuracy.rmse(predictions)

In [ ]:
def user_recommend(user_id, n=10):
    # List all unique book titles
    all_books = complete_df['Book-Title'].unique()

    # Remove books already rated by the user
    rated_books = complete_df[complete_df['User-ID'] == user_id]['Book-Title'].values
    books_to_predict = [book for book in all_books if book not in rated_books]

    # Predict ratings for remaining books
    predictions = []
    for book in books_to_predict:
        pred = model.predict(user_id, book)
        predictions.append((book, pred.est))

    # Sort predictions by estimated rating
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get top N recommendations
    top_n = predictions[:n]

    return top_n

In [ ]:
user_id = 271705
user_recommended = user_recommend(user_id)
print(f"Top 10 recommended books for user {user_id}:")
for i, (title, _) in enumerate(user_recommended, start=1):
    print(f"{i}. {title}")

In [ ]:
# Fill missing values in 'Age' with the median age
users['Age'].fillna(users['Age'].median(), inplace=True)

# Merge ratings with books
ratings_books = pd.merge(ratings, books, on='ISBN')

# Drop rows with missing values in important columns
ratings_books.dropna(subset=['Book-Title', 'Book-Author'], inplace=True)


In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Prepare data for Surprise
reader = Reader(rating_scale=(0, 10))
data = Dataset.load_from_df(ratings[['User-ID', 'ISBN', 'Book-Rating']], reader)

# Train-test split
trainset, testset = train_test_split(data, test_size=0.2)

# Use Singular Value Decomposition (SVD) algorithm
algo = SVD()

# Train the algorithm
algo.fit(trainset)

# Test the algorithm
predictions = algo.test(testset)

# Calculate RMSE
accuracy.rmse(predictions)


In [ ]:
all_books_clean = books.dropna()

# Sort books by ISBN then put them in the fix_books variable
fix_books = all_books_clean.sort_values('ISBN', ascending=True)

preparation = fix_books.drop_duplicates('ISBN')

# convert the 'ISBN' data series into list form
isbn_id = preparation['ISBN'].tolist()

# convert the 'Book-Title' data series into list form
book_title = preparation['Book-Title'].tolist()

# convert the 'Book-Author' data series into list form
book_author = preparation['Book-Author'].tolist()

# convert the 'Year-Of-Publication' data series into list form
year_of_publication = preparation['Year-Of-Publication'].tolist()

# convert the 'Publisher' data series into list form
publisher = preparation['Publisher'].tolist()

books_new = pd.DataFrame({
    'isbn': isbn_id,
    'book_title': book_title,
    'book_author': book_author,
    'year_of_publication': year_of_publication,
    'publisher': publisher

})

books_new

In [ ]:
#only the first 20,000 data points will be used
books_new = books_new[:20000]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

data = books_new

# Initialize TfidfVectorizer
tf = TfidfVectorizer()

# Perform IDF calculations on book_author data
tf.fit(['book_author'])

# Mapping array from integer index features to name features
tf.get_feature_names_out()

# Performs a fit and then transforms it into matrix form
tfidf_matrix = tf.fit_transform(data['book_author'])

# View the tfidf matrix size
tfidf_matrix.shape

tfidf_matrix.todense()

# Example usage
#print(get_content_recommendations('The Hobbit'))


In [ ]:
pd.DataFrame(
    tfidf_matrix.todense(),
    columns=tf.get_feature_names_out(),
    index=data.book_title
).sample(15, axis=1).sample(10, axis=0)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculating cosine similarity on the tf-idf matrix
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim

In [ ]:
# Create a dataframe from the cosine_sim variable with rows and columns in the form of book titles
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['book_title'], columns=data['book_title'])
print('Shape:', cosine_sim_df.shape)

# View the similarity matrix for each book title
cosine_sim_df.sample(5, axis=1).sample(10, axis=0)

In [ ]:
def get_content_recommendations(book_title, similarity_data=cosine_sim_df, items=data[['book_title', 'book_author']], k=5):
     # Retrieve data by using argpartition to partition indirectly along a given axis
     # Dataframe converted to numpy
     # Range(start, stop, step)
     index = similarity_data.loc[:,book_title].to_numpy().argpartition(range(-1, -k, -1))

     # Retrieve data with the greatest similarity from the existing index
     closest = similarity_data.columns[index[-1:-(k+2):-1]]

     # Drop book_title so that the name of the book you are looking for does not appear in the recommendation list
     closest = closest.drop(book_title, errors='ignore')

     return pd.DataFrame(closest).merge(items).head(k)

In [ ]:
book_title_test = "Entering the Silence : Becoming a Monk and a Writer (The Journals of Thomas Merton, V. 2)" # book title example

data[data.book_title.eq(book_title_test)]


In [ ]:
# Get recommendations for similar book titles
get_content_recommendations(book_title_test)

## Hybrid Recommender

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

class HybridRecommender:
    def __init__(self):
        # Load datasets
        self.books_df = pd.read_csv('Books.csv')
        self.ratings_df = pd.read_csv('Ratings.csv')
        self.users_df = pd.read_csv('Users.csv')

        # Initialize content-based features
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.book_features = None

        # Initialize collaborative filtering model
        self.cf_model = None

    def prepare_content_based_features(self):
        # Fill missing values with an empty string
        self.books_df['Book-Title'] = self.books_df['Book-Title'].fillna('')
        self.books_df['Book-Author'] = self.books_df['Book-Author'].fillna('')
        self.books_df['Publisher'] = self.books_df['Publisher'].fillna('')

        # Combine book features for content-based filtering
        self.books_df['content'] = self.books_df['Book-Title'] + ' ' + \
                                   self.books_df['Book-Author'] + ' ' + \
                                   self.books_df['Publisher']

        # Create TF-IDF matrix
        self.book_features = self.tfidf.fit_transform(self.books_df['content'])

    def get_content_based_recommendations(self, book_id, n_recommendations=5):
        # Get book index
        book_idx = self.books_df[self.books_df['ISBN'] == book_id].index[0]

        # Fit NearestNeighbors model
        nn_model = NearestNeighbors(n_neighbors=n_recommendations+1, metric='cosine', algorithm='brute')
        nn_model.fit(self.book_features)

        # Find similar books
        distances, indices = nn_model.kneighbors(self.book_features[book_idx], n_neighbors=n_recommendations+1)

        # Exclude the first one (itself)
        similar_books_indices = indices.flatten()[1:]

        return [self.books_df.iloc[i]['ISBN'] for i in similar_books_indices]

    def train_collaborative_filtering(self):
        # Prepare data for surprise library
        reader = Reader(rating_scale=(1, 10))
        data = Dataset.load_from_df(self.ratings_df[['User-ID', 'ISBN', 'Book-Rating']], reader)

        # Split data
        trainset, testset = train_test_split(data, test_size=0.2)

        # Train SVD model
        self.cf_model = SVD()
        self.cf_model.fit(trainset)

    def get_collaborative_recommendations(self, user_id, n_recommendations=5):
        # Get all books
        all_books = self.books_df['ISBN'].unique()

        # Get predictions for all books
        predictions = []
        for book_id in all_books:
            pred = self.cf_model.predict(user_id, book_id)
            predictions.append((book_id, pred.est))

        # Sort by predicted rating
        predictions.sort(key=lambda x: x[1], reverse=True)

        return [pred[0] for pred in predictions[:n_recommendations]]

    def get_hybrid_recommendations(self, user_id, book_id, n_recommendations=5, alpha=0.5):
        # Get content-based recommendations
        content_recs = self.get_content_based_recommendations(book_id, n_recommendations*2)

        # Get collaborative filtering recommendations
        cf_recs = self.get_collaborative_recommendations(user_id, n_recommendations*2)

        # Combine recommendations with weights
        combined_recs = {}
        for rec in content_recs:
            combined_recs[rec] = alpha
        for rec in cf_recs:
            if rec in combined_recs:
                combined_recs[rec] += (1 - alpha)
            else:
                combined_recs[rec] = (1 - alpha)

        # Sort and get top recommendations
        sorted_recs = sorted(combined_recs.items(), key=lambda x: x[1], reverse=True)
        return [rec[0] for rec in sorted_recs[:n_recommendations]]

    def train(self):
        print("Preparing content-based features...")
        self.prepare_content_based_features()

        print("Training collaborative filtering model...")
        self.train_collaborative_filtering()

        print("Training complete!")

# Example usage
if __name__ == "__main__":
    recommender = HybridRecommender()
    recommender.train()

    # Example: Get hybrid recommendations for user 1 and book with ISBN '0195153448'
    recommendations = recommender.get_hybrid_recommendations(1, '0195153448')
    print("Hybrid Recommendations:", recommendations)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

class HybridRecommender:
    def __init__(self):
        # Load datasets
        self.books_df = pd.read_csv('Books.csv')
        self.ratings_df = pd.read_csv('Ratings.csv')
        self.users_df = pd.read_csv('Users.csv')

        # Initialize content-based features
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.book_features = None

        # Initialize collaborative filtering model
        self.cf_model = None

    def prepare_content_based_features(self):
        # Fill missing values with an empty string
        self.books_df['Book-Title'] = self.books_df['Book-Title'].fillna('')
        self.books_df['Book-Author'] = self.books_df['Book-Author'].fillna('')
        self.books_df['Publisher'] = self.books_df['Publisher'].fillna('')

        # Combine book features for content-based filtering
        self.books_df['content'] = self.books_df['Book-Title'] + ' ' + \
                                   self.books_df['Book-Author'] + ' ' + \
                                   self.books_df['Publisher']

        # Create TF-IDF matrix
        self.book_features = self.tfidf.fit_transform(self.books_df['content'])

    def get_content_based_recommendations(self, book_title, n_recommendations=5):
        # Check if book_title exists in the Book-Title column
        if book_title not in self.books_df['Book-Title'].values:
            print(f"Book Title '{book_title}' not found in the dataset.")
            return []

        # Get book index
        book_idx = self.books_df[self.books_df['Book-Title'] == book_title].index[0]

        # Fit NearestNeighbors model
        nn_model = NearestNeighbors(n_neighbors=n_recommendations+1, metric='cosine', algorithm='brute')
        nn_model.fit(self.book_features)

        # Find similar books
        distances, indices = nn_model.kneighbors(self.book_features[book_idx], n_neighbors=n_recommendations+1)

        # Exclude the first one (itself)
        similar_books_indices = indices.flatten()[1:]

        return [self.books_df.iloc[i]['Book-Title'] for i in similar_books_indices]




    def train_collaborative_filtering(self):
        # Prepare data for surprise library
        reader = Reader(rating_scale=(1, 10))
        data = Dataset.load_from_df(self.ratings_df[['User-ID', 'ISBN', 'Book-Rating']], reader)

        # Split data
        trainset, testset = train_test_split(data, test_size=0.2)

        # Train SVD model
        self.cf_model = SVD()
        self.cf_model.fit(trainset)

    def get_collaborative_recommendations(self, user_id, n_recommendations=5):
        # Get all books
        all_books = self.books_df['ISBN'].unique()

        # Get predictions for all books
        predictions = []
        for book_id in all_books:
            pred = self.cf_model.predict(user_id, book_id)
            predictions.append((book_id, pred.est))

        # Sort by predicted rating
        predictions.sort(key=lambda x: x[1], reverse=True)

        # Get top n recommendations
        top_books_isbn = [pred[0] for pred in predictions[:n_recommendations]]

        # Return book titles
        return [self.books_df[self.books_df['ISBN'] == isbn]['Book-Title'].values[0] for isbn in top_books_isbn]


    def get_hybrid_recommendations(self, user_id, book_id, n_recommendations=5, alpha=0.5):
        # Get content-based recommendations
        content_recs = self.get_content_based_recommendations(book_id, n_recommendations*2)

        # Get collaborative filtering recommendations
        cf_recs = self.get_collaborative_recommendations(user_id, n_recommendations*2)

        # Combine recommendations with weights
        combined_recs = {}
        for rec in content_recs:
            combined_recs[rec] = alpha
        for rec in cf_recs:
            if rec in combined_recs:
                combined_recs[rec] += (1 - alpha)
            else:
                combined_recs[rec] = (1 - alpha)

        # Sort and get top recommendations
        sorted_recs = sorted(combined_recs.items(), key=lambda x: x[1], reverse=True)
        return [rec[0] for rec in sorted_recs[:n_recommendations]]


    def train(self):
        print("Preparing content-based features...")
        self.prepare_content_based_features()

        print("Training collaborative filtering model...")
        self.train_collaborative_filtering()

        print("Training complete!")

# Example usage
if __name__ == "__main__":
    recommender = HybridRecommender()
    recommender.train()

    # Example: Get hybrid recommendations for user 1 and book with ISBN '0195153448'
    recommendations = recommender.get_hybrid_recommendations(1, "Brave New World")
    print("Hybrid Recommendations:", recommendations)